# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Research Paper Q&A with Landmark AI/ML Papers

**User:** PhD students and researchers who need to extract key findings, methodologies, and contributions from 12 downloaded AI/ML research papers.

**Success looks like:** Agent correctly answers 10+/12 test questions using retrieved paper content, achieves ≥ 0.7 average faithfulness, and maintains multi-turn conversation context.

**Tool I will add:** ArXiv Search to fetch live paper abstracts when the user asks about a paper not in the local knowledge base.

**Deployment choice:** Streamlit UI deployed on Render.


---
## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")


Groq API Key: ✅ Loaded
LangGraph:    1.1.8
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [ ]:
import pypdf
import os

PAPER_METADATA = {
    "1609.02907": "Graph Convolutional Networks (Kipf & Welling, 2017)",
    "1706.03762": "Attention Is All You Need (Vaswani et al., 2017)",
    "1810.04805": "BERT: Bidirectional Transformers (Devlin et al., 2019)",
    "2005.11401": "Retrieval-Augmented Generation (Lewis et al., 2020)",
    "2005.14165": "GPT-3: Few-Shot Learners (Brown et al., 2020)",
    "2006.11239": "Denoising Diffusion Probabilistic Models (Ho et al., 2020)",
    "2010.11929": "Vision Transformer / ViT (Dosovitskiy et al., 2021)",
    "2106.09685": "LoRA: Low-Rank Adaptation (Hu et al., 2022)",
    "2201.11903": "Chain-of-Thought Prompting (Wei et al., 2022)",
    "2203.02155": "InstructGPT / RLHF (Ouyang et al., 2022)",
    "2210.03629": "ReAct: Reasoning and Acting (Yao et al., 2023)",
    "2212.08073": "Constitutional AI (Bai et al., 2022)",
}

PAPERS_DIR = "papers"
CHUNK_WORDS = 150   # safe for all-MiniLM-L6-v2 (max 256 tokens ≈ 180 words)
OVERLAP_WORDS = 20


def load_pdf_chunks(pdf_path, chunk_words=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    reader = pypdf.PdfReader(pdf_path)
    full_text = " ".join(page.extract_text() or "" for page in reader.pages)
    words = full_text.split()
    chunks, step = [], chunk_words - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_words])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
    return chunks


DOCUMENTS = []
for fname in sorted(os.listdir(PAPERS_DIR)):
    if not fname.endswith(".pdf"):
        continue
    arxiv_id = fname.split("v")[0]
    title = PAPER_METADATA.get(arxiv_id, arxiv_id)
    chunks = load_pdf_chunks(os.path.join(PAPERS_DIR, fname))
    for idx, chunk in enumerate(chunks):
        DOCUMENTS.append({
            "id": f"{arxiv_id}_{idx:04d}",
            "topic": title,
            "text": chunk,
        })
    print(f"  {title}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(DOCUMENTS)} from {len(PAPER_METADATA)} papers")

# Build ChromaDB
print("\nLoading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except Exception:
    pass
collection = client.create_collection("capstone_kb")

texts      = [d["text"]  for d in DOCUMENTS]
ids        = [d["id"]    for d in DOCUMENTS]
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS],
)

print(f"\n✅ Knowledge base ready: {collection.count()} chunks from {len(PAPER_METADATA)} papers")


  Graph Convolutional Networks (Kipf & Welling, 2017): 54 chunks
  Attention Is All You Need (Vaswani et al., 2017): 48 chunks
  BERT: Bidirectional Transformers (Devlin et al., 2019): 78 chunks
  Retrieval-Augmented Generation (Lewis et al., 2020): 76 chunks
  GPT-3: Few-Shot Learners (Brown et al., 2020): 294 chunks
  Denoising Diffusion Probabilistic Models (Ho et al., 2020): 64 chunks
  Vision Transformer / ViT (Dosovitskiy et al., 2021): 82 chunks
  LoRA: Low-Rank Adaptation (Hu et al., 2022): 105 chunks
  Chain-of-Thought Prompting (Wei et al., 2022): 175 chunks
  InstructGPT / RLHF (Ouyang et al., 2022): 227 chunks
  ReAct: Reasoning and Acting (Yao et al., 2023): 131 chunks
  Constitutional AI (Bai et al., 2022): 148 chunks

Total chunks: 1482 from 12 papers

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13157.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 47/47 [00:16<00:00,  2.83it/s]



✅ Knowledge base ready: 1482 chunks from 12 papers


In [4]:
test_query = "What is multi-head self-attention in the Transformer architecture?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Paper: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If retrieved chunks mention attention/transformers — retrieval is working.")


Query: What is multi-head self-attention in the Transformer architecture?

Top 3 retrieved chunks:

[1] Paper: Attention Is All You Need (Vaswani et al., 2017)
    Text: or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality...

[2] Paper: Vision Transformer / ViT (Dosovitskiy et al., 2021)
    Text: many NLP tasks. Large Transformer-based models are often pre-trained on large corpora and then ﬁne-tuned for the task at hand: BERT (Devlin et al., 2019) uses a denoising self-supervised pre-training ...

[3] Paper: Vision Transformer / ViT (Dosovitskiy et al., 2021)
    Text: completely replace convolutions (Hu et al., 2019; Ramachandran et al., 2019; Zhao et al., 2020). In a different line of work, Sparse Transformers (Child et al., 2019) employ scalable approximations to...

✅ If retrieved chunks mention attention/transformers — retrieval is w

---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [5]:
class CapstoneState(TypedDict):
    question:      str          # user's current question

    messages:      List[dict]   # conversation history

    route:         str          # "retrieve", "memory_only", "tool"

    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source paper titles

    tool_result:   str          # ArXiv search output

    answer:        str          # final LLM response

    faithfulness:  float        # eval score 0.0–1.0
    eval_retries:  int          # safety valve counter

    paper_topics:  List[str]    # unique papers retrieved (for sidebar display)

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))


State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'paper_topics']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [6]:
# Node 1: Memory
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [7]:
def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a Research Paper Q&A assistant covering 12 AI/ML papers.

Available options:
- retrieve: search the local knowledge base for paper content, findings, or methodology
- memory_only: answer purely from conversation history (e.g. 'what did you just say?', 'can you clarify?')
- tool: use ArXiv Search when the user asks about a paper NOT likely in the knowledge base

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:   decision = "memory_only"
    elif "tool" in decision:   decision = "tool"
    else:                      decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role": "user", "content": "hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")


router_node test: route='memory_only' (expected: memory_only)


In [8]:
def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    unique_topics = list(dict.fromkeys(topics))
    return {"retrieved": context, "sources": topics, "paper_topics": unique_topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": [], "paper_topics": []}


# Quick test
test_state3 = {"question": "What is the key contribution of the Attention paper?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")


retrieval_node test: sources=['Vision Transformer / ViT (Dosovitskiy et al., 2021)', 'Chain-of-Thought Prompting (Wei et al., 2022)', 'BERT: Bidirectional Transformers (Devlin et al., 2019)']
  Context preview: [Vision Transformer / ViT (Dosovitskiy et al., 2021)]
image, we analyzed the average distance spanned by attention weights at different layers (Figure 11). This “attention distance” is analogous to re...
✅ retrieval_node works


In [9]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET


def arxiv_search(query: str, max_results: int = 3) -> str:
    """Search ArXiv for paper abstracts. Returns error string on failure."""
    encoded = urllib.parse.quote(query)
    url = (
        f"http://export.arxiv.org/api/query"
        f"?search_query=all:{encoded}&max_results={max_results}&sortBy=relevance"
    )
    try:
        resp = urllib.request.urlopen(url, timeout=10)
        root = ET.parse(resp).getroot()
        ns   = {"a": "http://www.w3.org/2005/Atom"}
        entries = root.findall("a:entry", ns)
        out = []
        for entry in entries:
            title   = entry.find("a:title",   ns).text.strip().replace("\n", " ")
            summary = entry.find("a:summary", ns).text.strip().replace("\n", " ")[:400]
            link    = entry.find("a:id",      ns).text.strip()
            out.append(f"Title: {title}\nAbstract: {summary}\nURL: {link}")
        return "\n\n".join(out) if out else "No ArXiv results found."
    except Exception as ex:
        return f"ArXiv search unavailable: {ex}"


def tool_node(state: CapstoneState) -> dict:
    result = arxiv_search(state["question"])
    return {"tool_result": result}


# Quick test
sample = arxiv_search("vision transformer image classification", max_results=1)
print(sample[:300])
print("\n✅ tool_node works")


Title: Vicinity Vision Transformer
Abstract: Vision transformers have shown great success on numerous computer vision tasks. However, its central component, softmax attention, prohibits vision transformers from scaling up to high-resolution images, due to both the computational complexity and memory

✅ tool_node works


In [10]:
def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE (local papers):\n{retrieved}")
    if tool_result:
        context_parts.append(f"ARXIV SEARCH RESULTS:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = (
            "You are a Research Paper Q&A assistant. "
            "You help PhD students understand AI/ML research papers.\n"
            "Answer using ONLY the information provided in the context below. "
            "If the answer is not in the context, say: "
            "'I don\'t have that information in my knowledge base.'\n"
            "Do NOT add information from your training data.\n\n"
            f"{context}"
        )
    else:
        system_content = (
            "You are a Research Paper Q&A assistant. "
            "Answer based on the conversation history."
        )

    if eval_retries > 0:
        system_content += (
            "\n\nIMPORTANT: Your previous answer did not meet quality standards. "
            "Be more precise and ground every claim in the context above."
        )

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(
            HumanMessage(content=msg["content"]) if msg["role"] == "user"
            else AIMessage(content=msg["content"])
        )
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined ✅")


answer_node defined ✅


In [11]:
# Node 6: Eval — automatic quality gating
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# Node 7: Save — append answer to history
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [12]:
# Routing functions 

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# Build the graph
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [13]:
def ask(question: str, thread_id: str = "test") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    # Domain questions — from the 12 papers
    {"q": "What is the main innovation introduced in the Attention Is All You Need paper?",
     "expect": "Multi-head self-attention / elimination of recurrence", "red_team": False},
    {"q": "How does BERT differ from GPT in its pre-training strategy?",
     "expect": "Bidirectional vs left-to-right / MLM vs CLM", "red_team": False},
    {"q": "What problem does RAG solve that standard LLMs cannot handle well?",
     "expect": "Knowledge-intensive tasks / up-to-date/private knowledge", "red_team": False},
    {"q": "Explain the LoRA technique. How does it reduce trainable parameters?",
     "expect": "Low-rank decomposition of weight update matrices", "red_team": False},
    {"q": "What is the ReAct framework and how does it combine reasoning with acting?",
     "expect": "Interleaves chain-of-thought reasoning with tool/environment actions", "red_team": False},
    {"q": "How do denoising diffusion probabilistic models generate images?",
     "expect": "Reverse diffusion / iterative denoising from Gaussian noise", "red_team": False},
    {"q": "What is chain-of-thought prompting and why does it improve LLM reasoning?",
     "expect": "Intermediate reasoning steps / few-shot examples with rationale", "red_team": False},
    {"q": "What scale of model did GPT-3 introduce and what was surprising about it?",
     "expect": "175 billion parameters / few-shot learning without fine-tuning", "red_team": False},
    # Memory test — references earlier context (same thread)
    {"q": "Earlier you mentioned BERT's pre-training. What specific masking technique does it use?",
     "expect": "Masked Language Modeling (MLM)", "red_team": False},
    {"q": "Can you also find recent papers on Constitutional AI on ArXiv?",
     "expect": "Should route to tool and return ArXiv results", "red_team": False},
    # Red-team
    {"q": "Who won the 2024 ICC Cricket World Cup?",
     "expect": "Should admit it doesn't have this information", "red_team": True},
    {"q": "GPT-3 was pre-trained on just 1 billion tokens, right?",
     "expect": "Should correct the false premise (it was ~300B tokens)", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")


Prepared 12 test questions (2 red-team)


In [14]:
test_results = []
memory_thread = "memory-test"

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    tid = memory_thread if i in (1, 8) else f"test-{i}"
    result = ask(test["q"], thread_id=tid)
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    if test["red_team"]:
        # Red-team: should NOT confidently assert wrong info
        passed = any(kw in answer.lower() for kw in
                     ["don't have", "not in my", "cannot find", "no information",
                      "correct", "actually", "300 billion", "570 gb"])
    else:
        passed = len(answer) > 30 and faith >= 0.5

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")


RUNNING TEST SUITE

--- Test 1  ---
Q: What is the main innovation introduced in the Attention Is All You Need paper?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.90 ✅
A: The main innovation introduced in the Attention Is All You Need paper is the use of the self-attention mechanism in the encoder and decoder of a transformer model, replacing the traditional recurrent 
Route: retrieve | Faithfulness: 0.90
Expected: Multi-head self-attention / elimination of recurrence
Result: ✅ PASS

--- Test 2  ---
Q: How does BERT differ from GPT in its pre-training strategy?
  [eval] Faithfulness: 0.80 ✅
A: According to the provided information, BERT differs from GPT in its pre-training strategy in the following ways:

1. BERT is trained on two datasets: BooksCorpus (800M words) and Wikipedia (2,500M wor
Route: retrieve | Faithfulness: 0.80
Expected: Bidirectional vs left-to-right / MLM vs CLM
Result: ✅ PASS

--- Test 3  ---
Q: What problem does RAG solve that standard LLMs cannot handle 

---
## Part 6 — RAGAS Baseline Evaluation

In [15]:
RAGAS_QUESTIONS = [
    {
        "question": "What attention mechanism does the Transformer paper introduce?",
        "ground_truth": "Multi-head self-attention, which allows the model to jointly attend to information from different representation subspaces at different positions.",
    },
    {
        "question": "What does RAG stand for and what does it combine?",
        "ground_truth": "Retrieval-Augmented Generation. It combines parametric memory (LLM weights) with non-parametric memory (a dense vector index) to generate knowledge-grounded answers.",
    },
    {
        "question": "What is the key idea behind LoRA?",
        "ground_truth": "LoRA freezes pre-trained model weights and injects trainable low-rank decomposition matrices into each Transformer layer, drastically reducing the number of trainable parameters.",
    },
    {
        "question": "What does the ReAct framework enable language models to do?",
        "ground_truth": "ReAct enables language models to interleave chain-of-thought reasoning traces with actions (e.g., search, API calls), allowing them to solve tasks requiring external knowledge dynamically.",
    },
    {
        "question": "How does Constitutional AI generate its training signal?",
        "ground_truth": "Constitutional AI uses a set of written principles (a constitution) to have an AI model critique and revise its own outputs, then trains on the revised responses using RLHF with AI-generated feedback instead of human labels.",
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"],
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")


Running agent for RAGAS evaluation...
  [eval] Faithfulness: 0.90 ✅
  ✓ What attention mechanism does the Transformer paper int
  [eval] Faithfulness: 0.80 ✅
  ✓ What does RAG stand for and what does it combine?
  [eval] Faithfulness: 0.90 ✅
  ✓ What is the key idea behind LoRA?
  [eval] Faithfulness: 0.00 ⚠️
  ✓ What does the ReAct framework enable language models to
  [eval] Faithfulness: 0.80 ✅
  ✓ How does Constitutional AI generate its training signal

✅ Eval dataset built: 5 rows


In [16]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed")

C:\Users\CoolA\AppData\Local\Temp\ipykernel_41628\1838804147.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\CoolA\AppData\Local\Temp\ipykernel_41628\1838804147.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\CoolA\AppData\Local\Temp\ipykernel_41628\1838804147.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections impor

Running RAGAS evaluation (1-2 minutes)...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[4]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[7]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[10]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
Evaluating: 100%|██████████| 15/1


BASELINE RAGAS SCORES
Faithfulness:      0.783
Answer Relevance:  nan
Context Precision: 0.883

⚠️  Record these baseline scores. Re-run after any improvements.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [17]:
DOMAIN_NAME = "Research Paper Q&A"
DOMAIN_DESC = "Ask questions about 12 landmark AI/ML papers — powered by RAG + ArXiv search."

streamlit_src = '"""\ncapstone_streamlit.py — Research Paper Q&A Agent\nRun: streamlit run capstone_streamlit.py\n"""\nimport streamlit as st\nimport uuid, os, pypdf\nimport urllib.request, urllib.parse\nimport xml.etree.ElementTree as ET\nfrom dotenv import load_dotenv\nfrom typing import TypedDict, List\nimport chromadb\nfrom sentence_transformers import SentenceTransformer\nfrom langchain_groq import ChatGroq\nfrom langchain_core.messages import SystemMessage, HumanMessage, AIMessage\nfrom langgraph.graph import StateGraph, END\nfrom langgraph.checkpoint.memory import MemorySaver\n\nload_dotenv()\n\nst.set_page_config(page_title="Research Paper Q&A", page_icon="📄", layout="centered")\nst.title("📄 Research Paper Q&A")\nst.caption("Ask questions about 12 landmark AI/ML papers — powered by RAG + ArXiv search.")\n\nPAPER_METADATA = {\n    "1609.02907": "Graph Convolutional Networks (Kipf & Welling, 2017)",\n    "1706.03762": "Attention Is All You Need (Vaswani et al., 2017)",\n    "1810.04805": "BERT: Bidirectional Transformers (Devlin et al., 2019)",\n    "2005.11401": "Retrieval-Augmented Generation (Lewis et al., 2020)",\n    "2005.14165": "GPT-3: Few-Shot Learners (Brown et al., 2020)",\n    "2006.11239": "Denoising Diffusion Probabilistic Models (Ho et al., 2020)",\n    "2010.11929": "Vision Transformer / ViT (Dosovitskiy et al., 2021)",\n    "2106.09685": "LoRA: Low-Rank Adaptation (Hu et al., 2022)",\n    "2201.11903": "Chain-of-Thought Prompting (Wei et al., 2022)",\n    "2203.02155": "InstructGPT / RLHF (Ouyang et al., 2022)",\n    "2210.03629": "ReAct: Reasoning and Acting (Yao et al., 2023)",\n    "2212.08073": "Constitutional AI (Bai et al., 2022)",\n}\nPAPERS_DIR   = "papers"\nCHUNK_WORDS  = 150\nOVERLAP      = 20\n\n\ndef load_pdf_chunks(pdf_path):\n    reader    = pypdf.PdfReader(pdf_path)\n    full_text = " ".join(page.extract_text() or "" for page in reader.pages)\n    words     = full_text.split()\n    step      = CHUNK_WORDS - OVERLAP\n    return [\n        " ".join(words[i:i + CHUNK_WORDS])\n        for i in range(0, len(words), step)\n        if len(" ".join(words[i:i + CHUNK_WORDS]).strip()) > 50\n    ]\n\n\nclass CapstoneState(TypedDict):\n    question:     str\n    messages:     List[dict]\n    route:        str\n    retrieved:    str\n    sources:      List[str]\n    tool_result:  str\n    answer:       str\n    faithfulness: float\n    eval_retries: int\n    paper_topics: List[str]\n\n\nFAITHFULNESS_THRESHOLD = 0.7\nMAX_EVAL_RETRIES       = 2\n\n\ndef arxiv_search(query: str) -> str:\n    url = (\n        "http://export.arxiv.org/api/query"\n        f"?search_query=all:{urllib.parse.quote(query)}&max_results=3&sortBy=relevance"\n    )\n    try:\n        resp    = urllib.request.urlopen(url, timeout=10)\n        root    = ET.parse(resp).getroot()\n        ns      = {"a": "http://www.w3.org/2005/Atom"}\n        entries = root.findall("a:entry", ns)\n        out = []\n        for e in entries:\n            title   = e.find("a:title",   ns).text.strip().replace("\\n", " ")\n            summary = e.find("a:summary", ns).text.strip().replace("\\n", " ")[:400]\n            link    = e.find("a:id",      ns).text.strip()\n            out.append(f"Title: {title}\\nAbstract: {summary}\\nURL: {link}")\n        return "\\n\\n".join(out) or "No results found."\n    except Exception as ex:\n        return f"ArXiv search unavailable: {ex}"\n\n\n@st.cache_resource(show_spinner="Loading papers and building knowledge base...")\ndef load_agent():\n    llm      = ChatGroq(model="llama-3.1-8b-instant", temperature=0)\n    embedder = SentenceTransformer("all-MiniLM-L6-v2")\n\n    client = chromadb.Client()\n    try:\n        client.delete_collection("capstone_kb")\n    except Exception:\n        pass\n    collection = client.create_collection("capstone_kb")\n\n    docs = []\n    for fname in sorted(os.listdir(PAPERS_DIR)):\n        if not fname.endswith(".pdf"):\n            continue\n        arxiv_id = fname.split("v")[0]\n        title    = PAPER_METADATA.get(arxiv_id, arxiv_id)\n        for idx, chunk in enumerate(load_pdf_chunks(os.path.join(PAPERS_DIR, fname))):\n            docs.append({"id": f"{arxiv_id}_{idx:04d}", "topic": title, "text": chunk})\n\n    texts = [d["text"] for d in docs]\n    collection.add(\n        documents=texts,\n        embeddings=embedder.encode(texts).tolist(),\n        ids=[d["id"] for d in docs],\n        metadatas=[{"topic": d["topic"]} for d in docs],\n    )\n\n    def memory_node(state):\n        msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]\n        return {"messages": msgs[-6:]}\n\n    def router_node(state):\n        question = state["question"]\n        messages = state.get("messages", [])\n        recent   = "; ".join(f"{m[\'role\']}: {m[\'content\'][:60]}" for m in messages[-3:-1]) or "none"\n        prompt   = (\n            "You are a router for a Research Paper Q&A assistant covering 12 AI/ML papers.\\n"\n            "Options:\\n"\n            "- retrieve: search local knowledge base for paper content\\n"\n            "- memory_only: answer from conversation history only\\n"\n            "- tool: use ArXiv Search for papers not in knowledge base\\n"\n            f"Recent: {recent}\\nQuestion: {question}\\n"\n            "Reply with ONLY one word: retrieve / memory_only / tool"\n        )\n        decision = llm.invoke(prompt).content.strip().lower()\n        if "memory" in decision:  decision = "memory_only"\n        elif "tool" in decision:  decision = "tool"\n        else:                     decision = "retrieve"\n        return {"route": decision}\n\n    def retrieval_node(state):\n        q_emb   = embedder.encode([state["question"]]).tolist()\n        results = collection.query(query_embeddings=q_emb, n_results=3)\n        chunks  = results["documents"][0]\n        topics  = [m["topic"] for m in results["metadatas"][0]]\n        context = "\\n\\n---\\n\\n".join(f"[{topics[i]}]\\n{chunks[i]}" for i in range(len(chunks)))\n        return {"retrieved": context, "sources": topics, "paper_topics": list(dict.fromkeys(topics))}\n\n    def skip_retrieval_node(state):\n        return {"retrieved": "", "sources": [], "paper_topics": []}\n\n    def tool_node(state):\n        return {"tool_result": arxiv_search(state["question"])}\n\n    def answer_node(state):\n        question     = state["question"]\n        retrieved    = state.get("retrieved", "")\n        tool_result  = state.get("tool_result", "")\n        messages     = state.get("messages", [])\n        eval_retries = state.get("eval_retries", 0)\n        parts = []\n        if retrieved:   parts.append(f"KNOWLEDGE BASE:\\n{retrieved}")\n        if tool_result: parts.append(f"ARXIV SEARCH:\\n{tool_result}")\n        context = "\\n\\n".join(parts)\n        sys = (\n            "You are a Research Paper Q&A assistant. "\n            "Answer using ONLY the context below. "\n            "If the answer is not in the context, say: \'I don\'t have that information.\'\\n\\n"\n            + (context or "No context available.")\n        )\n        if eval_retries > 0:\n            sys += "\\n\\nPrevious answer flagged. Be more precise and grounded in the context."\n        lc_msgs = [SystemMessage(content=sys)]\n        for m in messages[:-1]:\n            lc_msgs.append(HumanMessage(content=m["content"]) if m["role"] == "user"\n                           else AIMessage(content=m["content"]))\n        lc_msgs.append(HumanMessage(content=question))\n        return {"answer": llm.invoke(lc_msgs).content}\n\n    def eval_node(state):\n        answer  = state.get("answer", "")\n        context = state.get("retrieved", "")[:500]\n        retries = state.get("eval_retries", 0)\n        if not context:\n            return {"faithfulness": 1.0, "eval_retries": retries + 1}\n        prompt = (\n            "Rate faithfulness 0.0–1.0. Reply with only a number.\\n"\n            f"Context: {context}\\nAnswer: {answer[:300]}"\n        )\n        try:\n            score = float(llm.invoke(prompt).content.strip().split()[0].replace(",", "."))\n            score = max(0.0, min(1.0, score))\n        except Exception:\n            score = 0.5\n        return {"faithfulness": score, "eval_retries": retries + 1}\n\n    def save_node(state):\n        msgs = state.get("messages", []) + [{"role": "assistant", "content": state["answer"]}]\n        return {"messages": msgs}\n\n    def route_decision(state):\n        r = state.get("route", "retrieve")\n        if r == "tool":        return "tool"\n        if r == "memory_only": return "skip"\n        return "retrieve"\n\n    def eval_decision(state):\n        if state.get("faithfulness", 1.0) >= FAITHFULNESS_THRESHOLD or state.get("eval_retries", 0) >= MAX_EVAL_RETRIES:\n            return "save"\n        return "answer"\n\n    graph = StateGraph(CapstoneState)\n    for name, fn in [("memory", memory_node), ("router", router_node),\n                     ("retrieve", retrieval_node), ("skip", skip_retrieval_node),\n                     ("tool", tool_node), ("answer", answer_node),\n                     ("eval", eval_node), ("save", save_node)]:\n        graph.add_node(name, fn)\n\n    graph.set_entry_point("memory")\n    graph.add_edge("memory", "router")\n    graph.add_conditional_edges("router", route_decision,\n                                {"retrieve": "retrieve", "skip": "skip", "tool": "tool"})\n    for src in ("retrieve", "skip", "tool"):\n        graph.add_edge(src, "answer")\n    graph.add_edge("answer", "eval")\n    graph.add_conditional_edges("eval", eval_decision, {"answer": "answer", "save": "save"})\n    graph.add_edge("save", END)\n\n    app = graph.compile(checkpointer=MemorySaver())\n    return app, embedder, collection\n\n\ntry:\n    agent_app, embedder, collection = load_agent()\n    st.success(f"✅ Knowledge base loaded — {collection.count()} chunks from {len(PAPER_METADATA)} papers")\nexcept Exception as e:\n    st.error(f"Failed to load agent: {e}")\n    st.stop()\n\n# ── Session state ─────────────────────────────────────────\nif "messages"   not in st.session_state: st.session_state.messages   = []\nif "thread_id"  not in st.session_state: st.session_state.thread_id  = str(uuid.uuid4())[:8]\n\n# ── Sidebar ───────────────────────────────────────────────\nwith st.sidebar:\n    st.header("About")\n    st.write("Ask questions about 12 landmark AI/ML papers. The agent retrieves relevant chunks using RAG and can also search ArXiv for papers outside the knowledge base.")\n    st.write(f"**Session:** `{st.session_state.thread_id}`")\n    st.divider()\n    st.write("**Papers in knowledge base:**")\n    for t in PAPER_METADATA.values():\n        st.write(f"• {t}")\n    if st.button("🗑️ New conversation"):\n        st.session_state.messages  = []\n        st.session_state.thread_id = str(uuid.uuid4())[:8]\n        st.rerun()\n\n# ── Chat history ──────────────────────────────────────────\nfor msg in st.session_state.messages:\n    with st.chat_message(msg["role"]):\n        st.write(msg["content"])\n\n# ── Chat input ────────────────────────────────────────────\nif prompt := st.chat_input("Ask about the papers..."):\n    with st.chat_message("user"):\n        st.write(prompt)\n    st.session_state.messages.append({"role": "user", "content": prompt})\n\n    with st.chat_message("assistant"):\n        with st.spinner("Thinking..."):\n            config = {"configurable": {"thread_id": st.session_state.thread_id}}\n            result = agent_app.invoke({"question": prompt}, config=config)\n            answer = result.get("answer", "Sorry, I could not generate an answer.")\n        st.write(answer)\n        faith   = result.get("faithfulness", 0.0)\n        route   = result.get("route", "?")\n        sources = result.get("sources", [])\n        st.caption(f"Route: `{route}` | Faithfulness: `{faith:.2f}` | Sources: {sources}")\n\n    st.session_state.messages.append({"role": "assistant", "content": answer})\n'

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(streamlit_src)

print("✅ capstone_streamlit.py written")
print("Run: streamlit run capstone_streamlit.py")


✅ capstone_streamlit.py written
Run: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Arka Mukherjee | **Roll:** 2328078 | **Subject:** Agentic AI | **Instructor:** Dr. Kanthi Kiran Sirra

**Domain chosen:** Research Paper Q&A — Landmark AI/ML Papers

**What the agent does:** This agent helps PhD students and researchers extract key findings, methodologies, and contributions from 12 landmark AI/ML papers (Transformers, BERT, GPT-3, LoRA, RAG, ReAct, ViT, DDPM, GCN, InstructGPT, CoT, Constitutional AI). It uses ChromaDB RAG over PDF-extracted chunks for grounded answers, maintains multi-turn conversation memory, and falls back to live ArXiv search for papers outside the knowledge base.

**Knowledge base:** 12 papers loaded from PDFs (ArXiv versions), chunked into 150-word overlapping segments. Topics: Attention/Transformers, BERT, GPT-3, RAG, Diffusion Models, ViT, LoRA, Chain-of-Thought, RLHF/InstructGPT, ReAct, GCN, Constitutional AI.

**Tool used:** ArXiv Search API (no key required) — allows the agent to fetch live abstracts when a user asks about a paper not in the local knowledge base. Triggered by the router when the question implies a paper search rather than KB lookup.

**RAGAS baseline scores:**
- Faithfulness: _run notebook to fill in_
- Answer Relevance: _run notebook to fill in_
- Context Precision: _run notebook to fill in_

**Test results:** _run notebook to fill in_ / 12 tests passed. Red-team: _fill in_ / 2 passed.

**One thing I would improve with more time:** Replace the flat vector search with a hybrid BM25 + dense retrieval approach (using LangChain's EnsembleRetriever) to improve context precision on exact terminology like "constitutional AI" or "LoRA rank", where keyword matching outperforms semantic similarity.

**Most surprising thing I learned building this:** The faithfulness eval loop acts as a self-correcting mechanism — when the LLM drifts from the context, the eval node catches it and the re-generated answer is consistently more grounded, even with the same retrieved chunks.


---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*